In [0]:
%sql
CREATE VOLUME IF NOT EXISTS f1.bronze.checkpoints;

In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/f1/bronze/landing/openf1"
CHECKPOINTS = "/Volumes/f1/bronze/checkpoints"

def load_bronze(endpoint):
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiline", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/{endpoint}/schema")
        .load(f"{LANDING}/{endpoint}")
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .writeStream
        .option("checkpointLocation", f"{CHECKPOINTS}/{endpoint}/checkpoint")
        .trigger(availableNow=True)                            # process new files, then stop
        .toTable(f"f1.bronze.openf1_{endpoint}")
        .awaitTermination())
        

In [0]:
ENDPOINTS = ["meetings", "sessions", "drivers", "session_result", "starting_grid",
             "laps", "stints", "pit", "weather", "race_control"]

for endpoint in ENDPOINTS:
    try:
        load_bronze(endpoint)
        n = spark.table(f"f1.bronze.openf1_{endpoint}").count()
        print(f"{endpoint:15} {n:8,} rows")
    except Exception as e:
        print(f"{endpoint:15} FAILED: {str(e)[:150]}")